In [ ]:
import sys
sys.path.insert(0, "..")
from src.model import load_model, chat

In [ ]:
model, tokeniser = load_model()
chat(model, tokeniser, "Explain compound interest in two sentences")

In [ ]:
import time

neutral = "What's 17% of 340?"
stressed = "I've been up all night stressing about money and I can't think straight. What's 17% of 340?"

for label, prompt in [("neutral", neutral), ("stressed", stressed)]:
    t0 = time.time()
    reply = chat(model, tokeniser, prompt, max_new_tokens=80)
    secs = time.time() - t0
    n_tokens = len(tokeniser(reply)["input_ids"])
    print(f"=== {label} | {secs:.1f}s | {n_tokens} tokens | thinking block: {'<think>' in reply} ===")
    print(reply)
    print()

In [ ]:
import torch
from src.hooks import get_residual_activations

acts_n = get_residual_activations(model, tokeniser, neutral)
acts_s = get_residual_activations(model, tokeniser, stressed)
print("shape:", acts_n.shape)   # expect [n_layers, hidden]

# norm ramp across layers
print("norms:", [round(float(x), 1) for x in acts_n.norm(dim=1)])

# cosine similarity per layer between neutral and stressed (same question)
cos = torch.nn.functional.cosine_similarity(acts_n, acts_s, dim=1)
print("cos(neutral, stressed):", [round(float(c), 3) for c in cos])

# control: two neutral prompts with different questions
other = "What's the capital of Australia?"
acts_o = get_residual_activations(model, tokeniser, other)
cos2 = torch.nn.functional.cosine_similarity(acts_n, acts_o, dim=1)
print("cos(neutral, other question):", [round(float(c), 3) for c in cos2])

In [ ]:
pairs = [
    ("What's 17% of 340?", "I've been up all night stressing about money and I can't think straight. What's 17% of 340?"),
    ("Explain how compound interest works.", "I'm panicking about my loans and can't focus. Explain how compound interest works."),
    ("What's the capital of Australia?", "Honestly I'm falling apart today. What's the capital of Australia?"),
    ("How do I reverse a list in Python?", "Everything's going wrong and I'm so anxious. How do I reverse a list in Python?"),
    ("Is it true we only use 10% of our brains?", "I've had the worst week and I'm barely holding it together. Is it true we only use 10% of our brains?"),
]

L = 18  # a middle-late layer
neu = torch.stack([get_residual_activations(model, tokeniser, n)[L] for n, s in pairs])
stv = torch.stack([get_residual_activations(model, tokeniser, s)[L] for n, s in pairs])

direction = (stv.mean(0) - neu.mean(0))
direction = direction / direction.norm()

# how does it compare to noise?
rand = torch.randn_like(direction); rand = rand / rand.norm()
print("cos(direction, random):", float(torch.dot(direction, rand)))

# does each stressed prompt project higher than its neutral twin?
proj_n = (neu - neu.mean(0)) @ direction
proj_s = (stv - stv.mean(0)) @ direction
print("neutral projections:", [round(float(x),1) for x in proj_n])
print("stressed projections:", [round(float(x),1) for x in proj_s])

In [ ]:
from src.hooks import steer_generate

prompt = "What's the capital of Australia?"
for N in [0, 3, 6, 10]:
    out = steer_generate(model, tokeniser, prompt, direction, layers=[L], N=N, max_new_tokens=60)
    print(f"--- N={N} ---\n{out}\n")

# control: random direction at the strongest scale
out = steer_generate(model, tokeniser, prompt, rand, layers=[L], N=10, max_new_tokens=60)
print(f"--- random, N=10 ---\n{out}\n")

In [ ]:
layers = list(range(12, 24))
for N in [0, 50, 100, 200, 400]:
    out = steer_generate(model, tokeniser, prompt, direction, layers=layers, N=N, max_new_tokens=60)
    print(f"--- N={N} ---\n{out}\n")

out = steer_generate(model, tokeniser, prompt, rand, layers=layers, N=200, max_new_tokens=60)
print(f"--- random, N=200 ---\n{out}\n")

In [ ]:
for N in [8, 15, 25, 35]:
    out = steer_generate(model, tokeniser, prompt, direction, layers=layers, N=N, max_new_tokens=60)
    print(f"--- N={N} ---\n{out}\n")